# Очистка таблиц Users, Items и Users_Items

В этом ноутбуке я привожу таблицы в нормальный вид, оставляю только нужные столбцы и строки.

## Импорт библиотек

In [37]:
import pandas as pd
import numpy as np
import os
import re
import scipy.stats as sts
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from pathlib import Path

## Загрузка данных

In [38]:
BASE_PATH = os.path.abspath("../../../Tables/BaseTable")

In [39]:
users_df = pd.read_csv(os.path.join(BASE_PATH, 'Users.csv'), sep=';', encoding='cp1251')
items_df = pd.read_csv(os.path.join(BASE_PATH, 'Items.csv'), sep=';', encoding='cp1251')
user_item_df = pd.read_csv(os.path.join(BASE_PATH,  'User_items_05_12_2025_04_03_2026.csv'), sep=';', encoding='cp1251')
user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')

C:\Users\msgt0\AppData\Local\Temp\ipykernel_4480\4085756075.py:4: DtypeWarning: Columns (0: 	Телефон ) have mixed types. Specify dtype option on import or set low_memory=False.
  user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')


In [40]:
print("Изначальные размеры:")
print(f"Users: {users_df.shape}")
print(f"Items: {items_df.shape}")
print(f"User_Items: {user_item_df.shape}")
print(f"User_Item_year: {user_item_year_df.shape}")

Изначальные размеры:
Users: (2893, 18)
Items: (258, 8)
User_Items: (8762, 17)
User_Item_year: (39289, 19)


In [41]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 18 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Имя                                        2892 non-null   str    
 1   Телефон                                    2892 non-null   str    
 2   Email                                      997 non-null    str    
 3   Категории                                  58 non-null     str    
 4   Дата рождения                              2 non-null      str    
 5   Потратил, ?                                2892 non-null   str    
 6   Оплатил, ?                                 2892 non-null   str    
 7   Пол                                        6 non-null      str    
 8   Карта                                      0 non-null      float64
 9   Скидка                                     2892 non-null   float64
 10  Последний визит                    

In [42]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


In [43]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8762 entries, 0 to 8761
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    5000 non-null   str    
 1   Специализация мастера           5000 non-null   str    
 2   Имя клиента                     3138 non-null   str    
 3   	Телефон                        3138 non-null   float64
 4   Email                           1361 non-null   str    
 5   Время визита                    5000 non-null   str    
 6   Имя	 создателя                  5000 non-null   str    
 7   Дата создания                   5000 non-null   str    
 8   Статус                          5000 non-null   str    
 9   Источник                        5000 non-null   str    
 10  Категория услуги                8663 non-null   str    
 11  Название услуги                 8663 non-null   str    
 12  Стоимость, руб                  8663 non-null

In [44]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

Заранее изучив данные, выявлены необходимые операции:

__Для Users__

1) Вместо Имя+Телефон+Email ввести `id_user`, а Email удалить
2) Удалить столбцы Дата рождения, Потратил в руб; Карта; Скидка; Последний визит; Чаевые; Количество посещений; Комментарий; Дополнительный телефон; Согласен на получение рассылок; Согласен на обработку персональных данных т.к. они большинством своим не информативны либо полностью пустые
3) В столбце Категория есть несколько возможных значений я хочу:
- все пропуски и значения "Клиенты отщепенцев", "ПРЕДОПЛАТА" заменить на "обычный"
- удалить все экземпляры (строчки), где категория "Черный список"
- заменить значение "Постоянный" на "обычный" (т.к. в таблице только 1 такой экземпляр)
4) Удалить пропуски, если они останутся

__Для Items__

1) Удалить все услуги, где значение "Категория товара или услуги в продаже" = "Солярий" (слишком много посещений и направление никак не связано с остальными услугами)
2) Удалить все услуги, которые были оказаны слишком маленькое количество раз ("Количество оказанных услуг"<8 раз) или где количество уникальных клиентов слишком маленькое ("Уникальные клиенты" < 7).
3) По сути каждая строчка это и есть информация об уникальной услуге, но я бы добавил столбец `id_item` и сделал его PK

__Для User-Items__

1) Удалить все элементы связанные с солярием
2) Заменить имена столбцов на отношение сущьностей (то есть столбец "Имя", который относиться к клиенту назвать "Имя клиента" и т.п.)
3) Удалить Создал (Имя, Дата);Статус; Источник; Информация (Статус, Имя, Дата); Оплачено полностью; Комментарий; Категория (они не имеют смысла или нулевые)
4) Добавить `user_id` (для каждого клиента такой же как в таблице Users(то есть чтобы одному клиенту и там и там значился 1 и тот же id)), добавить `item_id` (по аналогии с `user_id`) и удалить "Email"
5) Объеденить некоторые услуги исходя из бизнесс логики 

__Для User-Items-Year__

1) Повторить pipline для User-Items, но на файле с годом  


## Обработка Users

#### Вводим user_id для каждого уникального пользователя, определяя его с помощью тройки Имя+телефон+Email

In [45]:
def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone_str = str(phone).replace('.0', '').replace('+7', '8')
    return phone_str.strip()

In [46]:
users_df["Телефон"] = users_df["Телефон"].apply(clean_phone)
users_df = users_df.drop_duplicates(subset=["Телефон"], keep="first")
users_df["id_user"] = pd.Categorical(users_df["Телефон"]).codes

#### Настройка столбца 'Категории'

In [47]:
users_df['Категории'] = users_df['Категории'].fillna('обычный')
users_df['Категории'] = users_df['Категории'].replace({
    'Клиенты отщепенцев': 'обычный',
    'ПРЕДОПЛАТА': 'обычный',
    'Постоянный': 'обычный'
})

In [48]:
users_df['Категории'].value_counts()

Категории
обычный          2881
Черный список      12
Name: count, dtype: int64

In [49]:
users_df = users_df[users_df['Категории'] != 'Черный список'].copy()
users_df['Категории'].value_counts()

Категории
обычный    2881
Name: count, dtype: int64

Оставляем столбец, т.к. в будущем возможно введение других категорий.

In [50]:
users_df = users_df[[
    'id_user', 'Имя', 'Телефон',
    'Категории', 'Оплатил, ?', 'Последний визит'
]].copy()
users_df = users_df.rename(columns={'Оплатил, ?': 'Оплачено в руб'})

In [51]:
users_df.info()

<class 'pandas.DataFrame'>
Index: 2881 entries, 0 to 2892
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2881 non-null   int16
 1   Имя              2880 non-null   str  
 2   Телефон          2880 non-null   str  
 3   Категории        2881 non-null   str  
 4   Оплачено в руб   2880 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 140.7 KB


In [52]:
users_df.head()

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
0,2706,Наталья,79859657657,обычный,50363,2026-03-04 13:00
1,462,дарья,79082451054,обычный,0,2026-03-04 13:00
2,965,Татьяна,79163219575,обычный,30880,2026-03-04 12:30
3,1924,Анастасия,79279021487,обычный,41470,2026-03-04 12:05
4,1206,Екатерина,79168230729,обычный,171450,2026-03-04 12:00


In [53]:
users_df[users_df['Телефон']=='79165161952']

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
121,1050,Арина Фомичева,79165161952,обычный,58040,2026-02-28 14:05


#### Удалим пропуски и зафиксируем таблицу

In [54]:
users_clean = users_df.dropna(subset=['Последний визит']).copy()

In [55]:
users_clean.isna().sum()

id_user            0
Имя                0
Телефон            0
Категории          0
Оплачено в руб     0
Последний визит    0
dtype: int64

In [56]:
users_clean.info()

<class 'pandas.DataFrame'>
Index: 2711 entries, 0 to 2719
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2711 non-null   int16
 1   Имя              2711 non-null   str  
 2   Телефон          2711 non-null   str  
 3   Категории        2711 non-null   str  
 4   Оплачено в руб   2711 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 132.4 KB


#### Настроем типы данных

In [57]:
users_clean['Оплачено в руб'] = users_clean['Оплачено в руб'].astype(str).str.replace(',', '.').astype(float)

In [58]:
users_clean['Последний визит'] = pd.to_datetime(users_clean['Последний визит'], errors='coerce')

In [59]:
users_clean.dtypes

id_user                     int16
Имя                           str
Телефон                       str
Категории                     str
Оплачено в руб            float64
Последний визит    datetime64[us]
dtype: object

## Обработка Items

In [60]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


# Новая предобработка

#### Переименовываем столбцы 

In [61]:
items_df.columns = items_df.columns.str.strip()  # убираем пробелы в именах
items_df = items_df.rename(columns={
    "Выручка от продажи услуг, ?": "Выручка от продажи услуг, руб",
    "Ср. выручка с одного клиента, ?": "Ср. выручка с одного клиента, руб"
})

#### Настраиваем типизацию

In [62]:
def clean_money_col(series):
    return pd.to_numeric(series.str.replace(" ", "").str.replace(",", "."), errors="coerce")

def clean_percent_col(series):
    # убираем %, пробелы, заменяем запятые, приводим к числу
    return pd.to_numeric(
        series.str.replace(" ", "")
           .str.replace("%", "")
           .str.replace(",", "."),
        errors="coerce"
    )

items_df["Количество оказанных услуг"] = pd.to_numeric(
    items_df["Количество оказанных услуг"], errors="coerce"
).astype("Int64")

items_df["Доля от оказанных услуг в %"] = clean_percent_col(items_df["Доля от оказанных услуг в %"])
items_df["Выручка от продажи услуг, руб"] = clean_money_col(items_df["Выручка от продажи услуг, руб"])
items_df["% от общей выручки за услуги"] = clean_percent_col(items_df["% от общей выручки за услуги"])
items_df["Ср. выручка с одного клиента, руб"] = clean_money_col(items_df["Ср. выручка с одного клиента, руб"])
items_df["Уникальные клиенты"] = pd.to_numeric(
    items_df["Уникальные клиенты"], errors="coerce"
).astype("Int64")

In [63]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Название услуги или товара             258 non-null    str    
 1   Категория товара или услуги в продаже  258 non-null    str    
 2   Количество оказанных услуг             258 non-null    Int64  
 3   Доля от оказанных услуг в %            258 non-null    float64
 4   Выручка от продажи услуг, руб          258 non-null    float64
 5   % от общей выручки за услуги           258 non-null    float64
 6   Ср. выручка с одного клиента, руб      258 non-null    float64
 7   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), str(2)
memory usage: 16.8 KB


Удалчем услуги с солярием 

In [64]:
items_df = items_df.loc[~items_df['Категория товара или услуги в продаже'].isin(['Солярий', 'Услуги ПО АБОНЕМЕНТАМ'])]

In [65]:
items_df['Категория товара или услуги в продаже'].unique()

<StringArray>
['Лазерная Эпиляция (FOR WOMEN)',                  'Косметология',
         'Парикмахерские услуги',             'Пирсинг (Проколы)',
             'Маникюр / Педикюр',     'Инъекционная косметология',
  'Татуаж / Перманентный Макияж',   'Лазерная Эпиляция (FOR MEN)',
            'Наращивание ресниц']
Length: 9, dtype: str

In [66]:
items_filtered = items_df[
    (items_df["Количество оказанных услуг"] >= 6) &
    (items_df["Уникальные клиенты"] >= 5)
].copy()

## Создаем items_dim

In [67]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [68]:
ITEM_COL = "Название услуги или товара"

items_df[ITEM_COL] = items_df[ITEM_COL].str.strip().str.lower().replace(rename_map)

In [69]:
items_df = items_df.groupby(ITEM_COL).agg({
    'Категория товара или услуги в продаже': 'first', # Берем первую попавшуюся категорию
    'Количество оказанных услуг': 'sum',            # Суммируем количество
    'Выручка от продажи услуг, руб': 'sum',         # Суммируем выручку
    'Уникальные клиенты': 'sum',                    # здесь может быть погрешность (один клиент мог брать обе услуги)
    # Для долей и процентов лучше пересчитать их заново после группировки, 
    # либо взять среднее, если это допустимо:
    'Доля от оказанных услуг в %': 'mean',
    '% от общей выручки за услуги': 'mean',
    'Ср. выручка с одного клиента, руб': 'mean'
}).reset_index()

#### Добавим id_item

In [70]:
items_dim_cols = [
    "Название услуги или товара",
    "Категория товара или услуги в продаже"
]

# присвоим id_item (сквозной integer, как в твоей схеме)
items_dim = (items_df[items_dim_cols]
    .reset_index(drop=True)
    .reset_index()  # index = 0..N-1
    .rename(columns={"index": "id_item"})
)

## Создание items_stats

In [71]:
items_stats_cols = [
    "id_item",
    "Название услуги или товара",
    "Категория товара или услуги в продаже",
    "Количество оказанных услуг",
    "Доля от оказанных услуг в %",
    "Выручка от продажи услуг, руб",
    "% от общей выручки за услуги",
    "Ср. выручка с одного клиента, руб",
    "Уникальные клиенты",
]

In [72]:
items_df["id_item"] = range(len(items_df))

items_stats = items_df[items_stats_cols].copy()

Упорядочиваем

In [73]:
items_stats = items_stats.sort_values("id_item").reset_index(drop=True)

In [74]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                235 non-null    int64  
 1   Название услуги или товара             235 non-null    str    
 2   Категория товара или услуги в продаже  235 non-null    str    
 3   Количество оказанных услуг             235 non-null    Int64  
 4   Доля от оказанных услуг в %            235 non-null    float64
 5   Выручка от продажи услуг, руб          235 non-null    float64
 6   % от общей выручки за услуги           235 non-null    float64
 7   Ср. выручка с одного клиента, руб      235 non-null    float64
 8   Уникальные клиенты                     235 non-null    Int64  
dtypes: Int64(2), float64(4), int64(1), str(2)
memory usage: 17.1 KB


## Обработка User-Items-Year

Удаляем пустые столбцы 

In [75]:
user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')

C:\Users\msgt0\AppData\Local\Temp\ipykernel_4480\3792654743.py:1: DtypeWarning: Columns (0: 	Телефон ) have mixed types. Specify dtype option on import or set low_memory=False.
  user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')


In [76]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [77]:
user_item_year_df = user_item_year_df.loc[:, ~user_item_year_df.columns.str.contains("^Unnamed")]

Почистить имена колонок 

In [78]:
user_item_year_df.columns = user_item_year_df.columns.str.strip().str.replace(r'\t','', regex=True)

In [79]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   Телефон                         12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя создателя                   22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [80]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

In [81]:
def clean_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )

Приведение типов данных 

In [82]:
user_item_year_df[COL_PRICE_DISCOUNT] = clean_money(user_item_year_df[COL_PRICE_DISCOUNT])
user_item_year_df[COL_VISIT_TIME] = pd.to_datetime(user_item_year_df[COL_VISIT_TIME], errors='coerce')

Удалим строки без услуг (пропуски)

In [83]:
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].notna()]
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].str.strip() != ""]

In [84]:
user_item_year_df.shape[0]

39075

Фильтрация по солярию и абонимаентам 

In [85]:
mask_not_solar = user_item_year_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_year_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_year_df = user_item_year_df[mask_not_solar & mask_not_sub].copy()

In [86]:
user_item_year_df.shape[0]

28857

In [87]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 0 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Имя мастера                     12436 non-null  str           
 1   Специализация мастера           12436 non-null  str           
 2   Имя клиента                     12379 non-null  str           
 3   Телефон                         12379 non-null  object        
 4   Email                           5490 non-null   str           
 5   Время визита                    12436 non-null  datetime64[us]
 6   Имя создателя                   12436 non-null  str           
 7   Дата создания                   12436 non-null  str           
 8   Статус                          12436 non-null  str           
 9   Источник                        12436 non-null  str           
 10  Категория услуги                28857 non-null  str           
 11  Название услуги   

Заполнение пропусков (полупстых строк)

In [88]:
#user_item_year_df = user_item_year_df.sort_values([COL_VISIT_TIME, COL_CLIENT_NAME], ascending=False)

In [89]:
fill_cols = [col for col in user_item_year_df.columns if col != COL_SERVICE]
user_item_year_df[fill_cols] = user_item_year_df[fill_cols].ffill(axis=0)

In [90]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 0 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Имя мастера                     28857 non-null  str           
 1   Специализация мастера           28857 non-null  str           
 2   Имя клиента                     28857 non-null  str           
 3   Телефон                         28857 non-null  object        
 4   Email                           28845 non-null  str           
 5   Время визита                    28857 non-null  datetime64[us]
 6   Имя создателя                   28857 non-null  str           
 7   Дата создания                   28857 non-null  str           
 8   Статус                          28857 non-null  str           
 9   Источник                        28857 non-null  str           
 10  Категория услуги                28857 non-null  str           
 11  Название услуги   

Ренейм (объединение item)

In [91]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [92]:
user_item_year_df[COL_SERVICE] = user_item_year_df[COL_SERVICE].str.strip().str.lower().replace(rename_map)

In [93]:
user_item_year_df.shape

(28857, 17)

Добавляем id_user

In [94]:
users_df["Телефон"] = users_df["Телефон"].astype(str).str.strip()
user_item_year_df[COL_PHONE] = user_item_year_df[COL_PHONE].astype(str).str.strip()

In [95]:
user_item_year_df["Телефон"] = user_item_year_df["Телефон"].apply(clean_phone)

In [96]:
user_item_year_df = user_item_year_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [97]:
user_item_year_df.isna().sum()

Имя мастера                        0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                             12
Время визита                       0
Имя создателя                      0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                       17
Категория                         12
id_user                           88
dtype: int64

In [98]:
user_item_year_df.shape

(28857, 18)

Удаляем пропуски по id_user

In [99]:
user_item_year_df = user_item_year_df[user_item_year_df["id_user"].notna() & user_item_year_df["Категория"].notna()].copy()
user_item_year_df["id_user"] = user_item_year_df["id_user"].astype("Int64")

In [100]:
user_item_year_df.shape

(28760, 18)

In [101]:
user_item_year_df.isna().sum()

Имя мастера                       0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             0
Время визита                      0
Имя создателя                     0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       5
Категория                         0
id_user                           0
dtype: int64

Удаляем неинформатиыные столбцы

In [102]:
user_item_year_df = user_item_year_df.drop(columns=['Email', 'Статус', 'Дата создания','Имя создателя',
                                           'Оплачено полностью', 'Комментарий', 'Источник', 'Телефон' ,
                                           'Стоимость с учетом скидки, руб'])
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28760 entries, 12 to 28856
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Имя мастера            28760 non-null  str           
 1   Специализация мастера  28760 non-null  str           
 2   Имя клиента            28760 non-null  str           
 3   Время визита           28760 non-null  datetime64[us]
 4   Категория услуги       28760 non-null  str           
 5   Название услуги        28760 non-null  str           
 6   Стоимость, руб         28760 non-null  float64       
 7   Категория              28760 non-null  str           
 8   id_user                28760 non-null  Int64         
dtypes: Int64(1), datetime64[us](1), float64(1), str(6)
memory usage: 2.2 MB


Добавление id_item

In [103]:
user_item_year_df['Название услуги'] = (user_item_year_df['Название услуги']
                                        .str.strip()
                                        .str.lower()
                                        .str.replace(r'\s+', ' ', regex=True))

In [104]:
items_dim['Название услуги или товара'] = (items_dim['Название услуги или товара']
                                           .str.strip()
                                           .str.lower()
                                           .str.replace(r'\s+', ' ', regex=True))

In [105]:
user_item_year_clean = pd.merge(
    user_item_year_df,
    items_dim[["Название услуги или товара", "id_item"]],
    left_on="Название услуги",
    right_on="Название услуги или товара",
    how="left"
)

In [106]:
user_item_year_clean = user_item_year_clean.drop(columns=['Название услуги или товара'])

In [107]:
user_item_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28760 entries, 0 to 28759
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Имя мастера            28760 non-null  str           
 1   Специализация мастера  28760 non-null  str           
 2   Имя клиента            28760 non-null  str           
 3   Время визита           28760 non-null  datetime64[us]
 4   Категория услуги       28760 non-null  str           
 5   Название услуги        28760 non-null  str           
 6   Стоимость, руб         28760 non-null  float64       
 7   Категория              28760 non-null  str           
 8   id_user                28760 non-null  Int64         
 9   id_item                28760 non-null  int64         
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 2.2 MB


In [108]:
f"Пропусков id_item: {user_item_year_clean['id_item'].isna().sum()}"

'Пропусков id_item: 0'

### user_item_df

In [109]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8762 entries, 0 to 8761
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    5000 non-null   str    
 1   Специализация мастера           5000 non-null   str    
 2   Имя клиента                     3138 non-null   str    
 3   	Телефон                        3138 non-null   float64
 4   Email                           1361 non-null   str    
 5   Время визита                    5000 non-null   str    
 6   Имя	 создателя                  5000 non-null   str    
 7   Дата создания                   5000 non-null   str    
 8   Статус                          5000 non-null   str    
 9   Источник                        5000 non-null   str    
 10  Категория услуги                8663 non-null   str    
 11  Название услуги                 8663 non-null   str    
 12  Стоимость, руб                  8663 non-null

Почистить имена колонок

In [110]:
user_item_df.columns = user_item_df.columns.str.strip().str.replace(r'\t','', regex=True)

In [111]:
user_item_df.columns

Index(['Имя мастера', 'Специализация мастера', 'Имя клиента', 'Телефон',
       'Email', 'Время визита', 'Имя создателя', 'Дата создания', 'Статус',
       'Источник', 'Категория услуги', 'Название услуги', 'Стоимость, руб',
       'Стоимость с учетом скидки, руб', 'Оплачено полностью', 'Комментарий',
       'Категория'],
      dtype='str')

In [112]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

Приводим типы данных

In [113]:
user_item_df[COL_PRICE_DISCOUNT] = clean_money(user_item_df[COL_PRICE_DISCOUNT])
user_item_df[COL_VISIT_TIME] = pd.to_datetime(user_item_df[COL_VISIT_TIME], errors='coerce')

In [114]:
user_item_df.shape

(8762, 17)

In [115]:
user_item_df = user_item_df[user_item_df[COL_SERVICE].notna()]
user_item_df = user_item_df[user_item_df[COL_SERVICE].str.strip() != ""]

In [116]:
user_item_df.shape

(8663, 17)

Фильтрация по солярию и абонимаентам 

In [117]:
mask_not_solar = user_item_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_df = user_item_df[mask_not_solar & mask_not_sub].copy()

In [118]:
user_item_df.shape

(6733, 17)

In [119]:
user_item_df.isna().sum()

Имя мастера                       3735
Специализация мастера             3735
Имя клиента                       3747
Телефон                           3747
Email                             5402
Время визита                      3735
Имя создателя                     3735
Дата создания                     3735
Статус                            3735
Источник                          3735
Категория услуги                     0
Название услуги                      0
Стоимость, руб                       0
Стоимость с учетом скидки, руб       0
Оплачено полностью                3735
Комментарий                       6022
Категория                         6314
dtype: int64

In [120]:
#user_item_df = user_item_df.sort_values([COL_VISIT_TIME, COL_CLIENT_NAME], ascending=False)

In [121]:
fill_cols = [col for col in user_item_df.columns if col != COL_SERVICE]
user_item_df[fill_cols] = user_item_df[fill_cols].ffill(axis=0)

In [122]:
user_item_df.isna().sum()

Имя мастера                        0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                              4
Время визита                       0
Имя создателя                      0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                       12
Категория                         37
dtype: int64

Ренейм (объеденяем item)

In [123]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [124]:
user_item_df[COL_SERVICE] = user_item_df[COL_SERVICE].str.strip().str.lower().replace(rename_map)

In [125]:
user_item_df.shape

(6733, 17)

Добавляем id_user

In [126]:
user_item_df[COL_PHONE] = user_item_df[COL_PHONE].astype(str).str.strip()

In [127]:
user_item_df["Телефон"] = user_item_df["Телефон"].apply(clean_phone)

In [128]:
user_item_df = user_item_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [129]:
user_item_df.isna().sum()

Имя мастера                        0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                              4
Время визита                       0
Имя создателя                      0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                       12
Категория                         37
id_user                           20
dtype: int64

Удаляем пропуски по id_user

In [130]:
user_item_df = user_item_df[user_item_df["id_user"].notna() & user_item_df["Категория"].notna()].copy()
user_item_df["id_user"] = user_item_df["id_user"].astype("Int64")

In [131]:
user_item_df['id_user'].isna().sum()

0

Удаляем не информатиыные столбцы

In [132]:
user_item_df = user_item_df.drop(columns=['Email', 'Статус', 'Дата создания','Имя создателя',
                                           'Оплачено полностью', 'Комментарий', 'Источник', 'Телефон' ,
                                           'Стоимость с учетом скидки, руб'])
user_item_df.info()

<class 'pandas.DataFrame'>
Index: 6679 entries, 37 to 6732
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Имя мастера            6679 non-null   str           
 1   Специализация мастера  6679 non-null   str           
 2   Имя клиента            6679 non-null   str           
 3   Время визита           6679 non-null   datetime64[us]
 4   Категория услуги       6679 non-null   str           
 5   Название услуги        6679 non-null   str           
 6   Стоимость, руб         6679 non-null   float64       
 7   Категория              6679 non-null   str           
 8   id_user                6679 non-null   Int64         
dtypes: Int64(1), datetime64[us](1), float64(1), str(6)
memory usage: 528.3 KB


Добавляем id_item

In [133]:
user_item_df['Название услуги'] = (user_item_df['Название услуги']
                                        .str.strip()
                                        .str.lower()
                                        .str.replace(r'\s+', ' ', regex=True))

In [134]:
user_item_clean = pd.merge(
    user_item_df,
    items_dim[["Название услуги или товара", "id_item"]],
    left_on="Название услуги",
    right_on="Название услуги или товара",
    how="left"
)

In [135]:
user_item_clean = user_item_clean.drop(columns=['Название услуги или товара'])

In [136]:
user_item_clean.isna().sum()

Имя мастера              0
Специализация мастера    0
Имя клиента              0
Время визита             0
Категория услуги         0
Название услуги          0
Стоимость, руб           0
Категория                0
id_user                  0
id_item                  0
dtype: int64

## Сортировка таблиц по id

In [137]:
# 1. users_clean по user_id
users_clean = users_clean.sort_values('id_user').reset_index(drop=True)
items_dim = items_dim.sort_values('id_item').reset_index(drop=True)
items_stats = items_stats.sort_values('id_item').reset_index(drop=True)
user_item_clean = user_item_clean.sort_values('id_user').reset_index(drop=True)
user_item_year_clean = user_item_year_clean.sort_values('id_user').reset_index(drop=True)

## Финальная информация о таблицах

In [138]:
users_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2711 entries, 0 to 2710
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_user          2711 non-null   int16         
 1   Имя              2711 non-null   str           
 2   Телефон          2711 non-null   str           
 3   Категории        2711 non-null   str           
 4   Оплачено в руб   2711 non-null   float64       
 5   Последний визит  2711 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int16(1), str(3)
memory usage: 111.3 KB


In [139]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 3 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                235 non-null    int64
 1   Название услуги или товара             235 non-null    str  
 2   Категория товара или услуги в продаже  235 non-null    str  
dtypes: int64(1), str(2)
memory usage: 5.6 KB


In [140]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                235 non-null    int64  
 1   Название услуги или товара             235 non-null    str    
 2   Категория товара или услуги в продаже  235 non-null    str    
 3   Количество оказанных услуг             235 non-null    Int64  
 4   Доля от оказанных услуг в %            235 non-null    float64
 5   Выручка от продажи услуг, руб          235 non-null    float64
 6   % от общей выручки за услуги           235 non-null    float64
 7   Ср. выручка с одного клиента, руб      235 non-null    float64
 8   Уникальные клиенты                     235 non-null    Int64  
dtypes: Int64(2), float64(4), int64(1), str(2)
memory usage: 17.1 KB


In [141]:
user_item_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28760 entries, 0 to 28759
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Имя мастера            28760 non-null  str           
 1   Специализация мастера  28760 non-null  str           
 2   Имя клиента            28760 non-null  str           
 3   Время визита           28760 non-null  datetime64[us]
 4   Категория услуги       28760 non-null  str           
 5   Название услуги        28760 non-null  str           
 6   Стоимость, руб         28760 non-null  float64       
 7   Категория              28760 non-null  str           
 8   id_user                28760 non-null  Int64         
 9   id_item                28760 non-null  int64         
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 2.2 MB


In [142]:
user_item_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 6679 entries, 0 to 6678
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Имя мастера            6679 non-null   str           
 1   Специализация мастера  6679 non-null   str           
 2   Имя клиента            6679 non-null   str           
 3   Время визита           6679 non-null   datetime64[us]
 4   Категория услуги       6679 non-null   str           
 5   Название услуги        6679 non-null   str           
 6   Стоимость, руб         6679 non-null   float64       
 7   Категория              6679 non-null   str           
 8   id_user                6679 non-null   Int64         
 9   id_item                6679 non-null   int64         
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 528.4 KB


## Выгрузка таблиц

In [143]:
SAVE_PATH = os.path.abspath("../../../Tables/CleanTable")

In [144]:
try:
    users_clean.to_csv( f"{SAVE_PATH}/users_clean.csv", index=False, encoding='utf-8')
    items_dim.to_csv(F"{SAVE_PATH}/items_dim.csv", index=False, encoding='utf-8')
    items_stats.to_csv(f"{SAVE_PATH}/items_stats.csv", index=False, encoding='utf-8')
    user_item_year_clean.to_csv(f"{SAVE_PATH}/user_items_year_clean.csv", index=False, encoding='utf-8')
    user_item_clean.to_csv(f"{SAVE_PATH}/user_item_clean.csv", index=False, encoding='utf-8')
except FileNotFoundError:
    print("Ошибка: Указанной папки не существует.")
except PermissionError:
    print("Ошибка: Файл занят другой программой или недостаточно прав.")
except OSError as e:
    print(f"Системная ошибка при записи: {e}")
